## Create a simple UI in Streamlit without using built-in layout functions, manually structuring components using HTML and CSS, and ensuring responsiveness across different screen sizes.

In [1]:
%pip install streamlit pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [4]:
import streamlit
import pandas
import sklearn
import joblib

print("All libraries installed successfully")

All libraries installed successfully


In [6]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna()

X = df.drop(columns=["target"])
y = df["target"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

joblib.dump(model, "heart_model_v20261013_160903.joblib")

print("Model saved successfully!")

Model saved successfully!


In [7]:
%cd "30oct folder ka complete path"
!streamlit run app.py

[WinError 2] The system cannot find the file specified: '30oct folder ka complete path'
C:\Users\Admin\heart_disease\2nd_nov


Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [8]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


## Task 1: Implement user input handling for prediction by validating inputs dynamically, handling edge cases such as missing or incorrect values, and optimizing input processing for real-time predictions.

### Step 1: `input.py` (validation + prediction + batch)

In [9]:
%%writefile input.py
"""
HeartCare - user input handling for prediction.

1. Dynamic validation : rules (allowed categories, number ranges) heart.csv se khud banti hain.
2. Edge cases         : missing, text, out-of-range, unknown category, inf/NaN, extra keys, wrong length.
3. Real-time speed    : ek hi predict_proba pass, repeated input ka cache, batch me vectorized validation.
"""
import numpy as np
import pandas as pd
from functools import lru_cache

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
MISSING_WORDS = {"", "na", "n/a", "nan", "null", "none", "?", "-", "unknown", "not known"}
MAX_MISSING = 3            # isse zyada features missing ho to prediction reject


# ---------------------------------------------------------------- 1. dynamic rules
def build_rules(df):
    """Training data se rules: <= 5 alag values wala column = category, baaki = number."""
    rules = {}
    for f in FEATURES:
        col = df[f].astype(float).dropna()
        vals = np.sort(col.unique())
        r = {"min": float(col.min()), "max": float(col.max())}
        if len(vals) <= 5:
            r.update(kind="category", allowed=vals.tolist(), fill=float(col.mode()[0]))
        else:
            span = col.max() - col.min()
            lo, hi = col.min() - 0.25 * span, col.max() + 0.25 * span     # impossible-value limit
            if col.min() >= 0:
                lo = max(lo, 0.0)                                          # negative nahi ho sakta
            r.update(kind="number", hard_lo=float(lo), hard_hi=float(hi), fill=float(col.median()))
        rules[f] = r
    return rules


def load_rules(csv_path="heart.csv"):
    return build_rules(pd.read_csv(csv_path).drop_duplicates().dropna())


# ---------------------------------------------------------------- 2. single input
def is_missing(v):
    if v is None or v is pd.NA:
        return True
    if isinstance(v, str):
        return v.strip().lower() in MISSING_WORDS
    return isinstance(v, (float, np.floating)) and np.isnan(v)


def to_number(v):
    if isinstance(v, (bool, int, float, np.integer, np.floating)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower().replace(",", ".")            # "2,3" bhi chalega
        return {"true": 1.0, "false": 0.0}.get(t) if t in ("true", "false") else float(t)
    raise TypeError("number nahi hai")


def validate_input(record, rules):
    """record = dict ya 13 values ki list. Return: {ok, x, errors, warnings, filled}"""
    out = {"ok": False, "x": None, "errors": [], "warnings": [], "filled": []}
    if isinstance(record, (list, tuple, np.ndarray)):
        if len(record) != len(FEATURES):
            out["errors"].append(f"{len(FEATURES)} values chahiye, {len(record)} mili")
            return out
        record = dict(zip(FEATURES, record))
    if not isinstance(record, dict):
        out["errors"].append("input dict ya list hona chahiye")
        return out

    rec = {str(k).strip().lower().replace(" ", "_"): v for k, v in record.items()}   # " Age " -> "age"
    extra = [k for k in rec if k not in rules]
    if extra:
        out["warnings"].append(f"extra fields ignore kiye: {extra}")

    x = np.zeros(len(FEATURES))
    for i, f in enumerate(FEATURES):
        v, r = rec.get(f), rules[f]
        if is_missing(v):
            x[i] = r["fill"]; out["filled"].append(f); continue
        try:
            val = to_number(v)
        except (ValueError, TypeError):
            out["errors"].append(f"{f}: '{v}' number nahi hai"); continue
        if not np.isfinite(val):
            out["errors"].append(f"{f}: inf/NaN allowed nahi"); continue
        if r["kind"] == "category":
            if val not in r["allowed"]:
                out["errors"].append(f"{f}: {val:g} allowed nahi (allowed: {[int(a) for a in r['allowed']]})")
        elif not r["hard_lo"] <= val <= r["hard_hi"]:
            out["errors"].append(f"{f}: {val:g} impossible value (limit {r['hard_lo']:.1f} se {r['hard_hi']:.1f})")
        elif not r["min"] <= val <= r["max"]:
            out["warnings"].append(f"{f}: {val:g} training range [{r['min']:g}, {r['max']:g}] ke bahar hai, prediction kam bharosemand")
        x[i] = val

    if len(out["filled"]) > MAX_MISSING:
        out["errors"].append(f"{len(out['filled'])} fields missing hain (max {MAX_MISSING} allowed)")
    elif out["filled"]:
        out["warnings"].append(f"missing fields median/mode se bhare gaye: {out['filled']}")
    if not out["errors"]:
        out["ok"], out["x"] = True, x
    return out


# ---------------------------------------------------------------- prediction (fast)
def predict_result(model, x):
    """x = 13 numbers. Pehle 2 pass (predict + predict_proba) the, ab sirf 1 pass."""
    probabilities = model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0]
    prediction = int(model.classes_[np.argmax(probabilities)])
    return prediction, probabilities


def make_predictor(model, size=1024):
    """Same input dobara aaye to model dobara nahi chalta (cache)."""
    @lru_cache(maxsize=size)
    def cached(key):
        return predict_result(model, list(key))
    return lambda x: cached(tuple(float(v) for v in x))


# ---------------------------------------------------------------- 3. batch (vectorized)
def _to_numbers(s):
    """Series -> (numbers, missing_mask). Text wali value NaN banti hai aur missing nahi maani jati."""
    if pd.api.types.is_numeric_dtype(s):
        num = s.astype(float).to_numpy()
        return num, np.isnan(num)
    txt = s.astype(str).str.strip().str.lower()
    miss = (s.isna() | txt.isin(MISSING_WORDS)).to_numpy()
    num = pd.to_numeric(txt.str.replace(",", ".", regex=False).replace({"true": "1", "false": "0"}), errors="coerce").to_numpy(dtype=float)
    return num, miss


def validate_batch(raw, rules):
    """DataFrame ki saari rows ek saath validate. Return: X, ok(bool array), reason(Series)"""
    raw = raw.rename(columns=lambda c: str(c).strip().lower().replace(" ", "_"))
    n = len(raw)
    X, bad, n_miss = np.zeros((n, len(FEATURES))), np.zeros(n, bool), np.zeros(n, int)
    reason = pd.Series([""] * n, index=raw.index, dtype=object)
    for i, f in enumerate(FEATURES):
        r = rules[f]
        if f in raw:
            num, miss = _to_numbers(raw[f])
        else:
            num, miss = np.full(n, np.nan), np.ones(n, bool)
        bad_f = ~miss & ~np.isfinite(num)                                 # text ya inf
        ok_num = ~miss & np.isfinite(num)
        if r["kind"] == "category":
            bad_f |= ok_num & ~np.isin(num, r["allowed"])
        else:
            bad_f |= ok_num & ((num < r["hard_lo"]) | (num > r["hard_hi"]))
        reason = reason + np.where(bad_f, f + " galat; ", "")
        bad |= bad_f; n_miss += miss
        X[:, i] = np.where(miss | bad_f, r["fill"], num)
    too_many = n_miss > MAX_MISSING
    reason = reason + np.where(too_many, "bahut fields missing; ", "")
    return X, ~(bad | too_many), reason


def predict_batch(model, rules, raw):
    """CSV/DataFrame ke liye: ek hi predict_proba call poori valid rows par."""
    X, ok, reason = validate_batch(raw, rules)
    out = pd.DataFrame({"status": np.where(ok, "ok", "rejected"), "reason": reason.str.strip("; ").to_numpy(),
                        "prediction": np.nan, "probability_%": np.nan}, index=raw.index)
    if ok.any():
        p = model.predict_proba(pd.DataFrame(X[ok], columns=FEATURES))
        out.loc[ok, "prediction"] = model.classes_[p.argmax(axis=1)]
        out.loc[ok, "probability_%"] = (p.max(axis=1) * 100).round(2)
    return out

Writing input.py


### Step 2: Dynamic rules 

In [10]:
import warnings; warnings.filterwarnings('ignore')
import joblib
import pandas as pd
from input import *

model = joblib.load('heart_model_v20261013_160903.joblib')
rules = load_rules('heart.csv')

pd.DataFrame(rules).T[['kind', 'min', 'max', 'hard_lo', 'hard_hi', 'allowed', 'fill']]

,kind,min,max,hard_lo,hard_hi,allowed,fill
age,number,29.0,77.0,17.0,89.0,NaN,55.5
sex,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",1.0
cp,category,0.0,3.0,NaN,NaN,"[0.0, 1.0, 2.0, 3.0]",0.0
trestbps,number,94.0,200.0,67.5,226.5,NaN,130.0
chol,number,126.0,564.0,16.5,673.5,NaN,240.5
fbs,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
restecg,category,0.0,2.0,NaN,NaN,"[0.0, 1.0, 2.0]",1.0
thalach,number,71.0,202.0,38.25,234.75,NaN,152.5
exang,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
oldpeak,number,0.0,6.2,0.0,7.75,NaN,0.8


### Step 3: Edge cases ka test
Har case ka expected result (`ok` ya `rejected`) pehle se likha hai. `match = True` ka matlab validator ne wahi kiya jo hona chahiye.

In [11]:
good = dict(pd.read_csv('heart.csv').iloc[0][FEATURES])        # ek sahi record
def with_(**kw):
    d = dict(good); d.update(kw); return d

cases = {
    'valid':                (good, True),
    'numbers as text':      ({k: str(v) for k, v in good.items()}, True),
    'comma decimal 2,3':    (with_(oldpeak='2,3'), True),
    'keys alag case/space': ({' Age ': 63, 'SEX': 1, **{k: v for k, v in good.items() if k not in ('age', 'sex')}}, True),
    '13 values ki list':    (list(good.values()), True),
    '1 missing (None)':     (with_(chol=None), True),
    '3 missing (mixed)':    (with_(chol='', trestbps='NA', thal=float('nan')), True),
    '5 missing':            (with_(chol=None, trestbps=None, thal=None, ca=None, slope=None), False),
    'empty dict':           ({}, False),
    'list too short':       ([1, 2], False),
    'not dict/list':        ('abc', False),
    'text in number':       (with_(age='sixty'), False),
    'age = 250':            (with_(age=250), False),
    'negative chol':        (with_(chol=-5), False),
    'inf value':            (with_(thalach=float('inf')), False),
    'unknown category':     (with_(cp=9), False),
    'category 1.5':         (with_(cp=1.5), False),
    'chol 600 (range ke bahar)': (with_(chol=600), True),
    'extra field':          ({**good, 'name': 'Riya'}, True),
}
rows = []
for name, (rec, expected) in cases.items():
    r = validate_input(rec, rules)
    rows.append((name, expected, r['ok'], expected == r['ok'], (r['errors'] + r['warnings'] or [''])[0][:60]))
res = pd.DataFrame(rows, columns=['case', 'expected_ok', 'got_ok', 'match', 'message'])
print('Matched:', res['match'].sum(), 'of', len(res))
res

Matched: 19 of 19


,case,expected_ok,got_ok,match,message
0,valid,True,True,True,
1,numbers as text,True,True,True,
2,"comma decimal 2,3",True,True,True,
3,keys alag case/space,True,True,True,
4,13 values ki list,True,True,True,
5,1 missing (None),True,True,True,missing fields median/mode se bhare gaye: ['ch...
6,3 missing (mixed),True,True,True,missing fields median/mode se bhare gaye: ['tr...
7,5 missing,False,False,True,5 fields missing hain (max 3 allowed)
8,empty dict,False,False,True,13 fields missing hain (max 3 allowed)
9,list too short,False,False,True,"13 values chahiye, 2 mili"


### Step 4: Batch (CSV) me galat rows pakadna
Ek messy DataFrame: kuch rows sahi, kuch me text, unknown category, impossible value, bahut zyada missing. `predict_batch` sirf valid rows par model chalata hai.

In [12]:
messy = pd.DataFrame([good, with_(age='abc'), with_(chol=None), with_(cp=7), with_(trestbps=900),
                      with_(thal=None, ca=None, slope=None, oldpeak=None), good])
pd.concat([messy[['age', 'cp', 'chol', 'trestbps']], predict_batch(model, rules, messy)], axis=1)

,age,cp,chol,trestbps,status,reason,prediction,probability_%
0,63.0,3.0,233.0,145.0,ok,,1.0,78.0
1,abc,3.0,233.0,145.0,rejected,age galat,NaN,NaN
2,63.0,3.0,NaN,145.0,ok,,1.0,78.0
3,63.0,7.0,233.0,145.0,rejected,cp galat,NaN,NaN
4,63.0,3.0,233.0,900.0,rejected,trestbps galat,NaN,NaN
5,63.0,3.0,233.0,145.0,rejected,bahut fields missing,NaN,NaN
6,63.0,3.0,233.0,145.0,ok,,1.0,78.0


### Step 5: Speed (real-time) check

In [13]:
import time
records = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].to_dict('records')[:100]

def old_way(rec):                                   # pehle jaisa: 2 pass, validation nahi
    d = pd.DataFrame([list(rec.values())], columns=FEATURES)
    return model.predict(d)[0], model.predict_proba(d)[0]

def new_way(rec):                                   # validation + 1 pass
    return predict_result(model, validate_input(rec, rules)['x'])

predictor = make_predictor(model)
def new_cached(rec):                                # validation + cache
    return predictor(validate_input(rec, rules)['x'])

def ms_per_call(fn):
    t = time.perf_counter(); [fn(r) for r in records]; return (time.perf_counter() - t) / len(records) * 1000

ms_per_call(old_way)                                # warm-up
speed = {'purana (2 pass, bina validation)': ms_per_call(old_way), 'naya: validation + 1 pass': ms_per_call(new_way)}
ms_per_call(new_cached); speed['naya + cache (same input dobara)'] = ms_per_call(new_cached)

big = pd.concat([pd.read_csv('heart.csv')[FEATURES]] * 100)       # 30,300 rows
t = time.perf_counter(); out = predict_batch(model, rules, big); speed['batch (30,300 rows, per row)'] = (time.perf_counter() - t) / len(big) * 1000

same = all(old_way(r)[0] == new_way(r)[0] for r in records)
print('Prediction purane tareeke jaisi hi hai:', same)
pd.Series(speed, name='ms per prediction').round(3)

Prediction purane tareeke jaisi hi hai: True


purana (2 pass, bina validation)    32.693
naya: validation + 1 pass           17.892
naya + cache (same input dobara)     0.148
batch (30,300 rows, per row)         0.016
Name: ms per prediction, dtype: float64

## Task 2: Display model predictions dynamically by implementing custom caching mechanisms to improve performance, ensuring smooth updates without unnecessary reloading, and handling concurrent user requests efficiently.

### Step 1: `cache.py`

In [16]:
%%writefile cache.py
"""
HeartCare - custom caching for predictions (Streamlit ka built-in cache nahi).

- LRU     : size limit; sabse purani entry hatti hai
- TTL     : entry kuch der baad expire (model/data purana na ho)
- Thread-safe : Lock se
- Single-flight : same input par 10 users ek saath aaye to model sirf 1 baar chalta hai,
                  baaki 9 wahi result ka wait karte hain (concurrent requests efficient)
"""
import time
import threading
from collections import OrderedDict


class _Call:
    """Jo computation chal rahi hai, uska result yahan rakhte hain."""
    def __init__(self):
        self.done = threading.Event()
        self.value = None
        self.error = None


class PredictionCache:

    def __init__(self, max_size=512, ttl=600):
        self.max_size = max_size
        self.ttl = ttl                      
        self._data = OrderedDict()         
        self._inflight = {}                
        self._lock = threading.Lock()
        self.hits = self.misses = self.waits = self.evictions = self.expired = 0

    def get_or_compute(self, key, compute):
        """Cache me ho to wahi do, warna compute() chalao (same key par ek hi baar)."""
        with self._lock:
            item = self._data.get(key)
            if item is not None:
                value, created = item
                if self.ttl is None or time.monotonic() - created < self.ttl:
                    self._data.move_to_end(key)         
                    self.hits += 1
                    return value
                del self._data[key]                    
                self.expired += 1

            call = self._inflight.get(key)
            owner = call is None
            if owner:
                call = self._inflight[key] = _Call()
                self.misses += 1
            else:
                self.waits += 1

        if owner:
            try:
                call.value = compute()                  
            except BaseException as error:
                call.error = error                      
            with self._lock:
                if call.error is None:
                    self._data[key] = (call.value, time.monotonic())
                    self._data.move_to_end(key)
                    while len(self._data) > self.max_size:
                        self._data.popitem(last=False)
                        self.evictions += 1
                del self._inflight[key]
            call.done.set()
        else:
            call.done.wait()

        if call.error is not None:
            raise call.error
        return call.value

    def clear(self):
        with self._lock:
            self._data.clear()

    def stats(self):
        with self._lock:
            total = self.hits + self.misses + self.waits
            return {"size": len(self._data), "hits": self.hits, "misses": self.misses,
                    "waits": self.waits, "evictions": self.evictions, "expired": self.expired,
                    "hit_rate_%": round(100 * (self.hits + self.waits) / total, 1) if total else 0.0}

Overwriting cache.py


### Step 2: Cache ka behaviour test (hit, LRU eviction, TTL)

In [17]:
import time
from cache import PredictionCache

small = PredictionCache(max_size=2, ttl=0.5)
runs = []
def work(v):
    runs.append(v); return v * 10

small.get_or_compute('a', lambda: work(1)); small.get_or_compute('a', lambda: work(1))     
small.get_or_compute('b', lambda: work(2)); small.get_or_compute('c', lambda: work(3))  
print('model/compute kitni baar chala:', len(runs), '(3 alag keys, 1 repeat)')
small.get_or_compute('a', lambda: work(1))                                                   
print('evicted key dobara compute hui:', runs.count(1) == 2)
time.sleep(0.6)
small.get_or_compute('c', lambda: work(3))                                                   

print(small.stats())

model/compute kitni baar chala: 3 (3 alag keys, 1 repeat)
evicted key dobara compute hui: True
{'size': 2, 'hits': 1, 'misses': 5, 'waits': 0, 'evictions': 2, 'expired': 1, 'hit_rate_%': 16.7}


### Step 3: Concurrent users ka test

In [18]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

patients = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values[:10]
requests = [patients[i % 10] for i in range(60)]

def predict_p1(x):
    return model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0, 1]

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: plain = list(pool.map(predict_p1, requests))
t_plain = time.perf_counter() - t

cache = PredictionCache(); runs = {'n': 0}
def cached_predict(x):
    def compute():
        runs['n'] += 1
        return predict_p1(x)
    return cache.get_or_compute(tuple(x), compute)

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: cached = list(pool.map(cached_predict, requests))
t_cache = time.perf_counter() - t

print(f'Bina cache : 60 model runs, {t_plain * 1000:.0f} ms')
print(f'Custom cache: {runs["n"]} model runs, {t_cache * 1000:.0f} ms')
print('Dono ke results same:', np.allclose(plain, cached))
print(cache.stats())

Bina cache : 60 model runs, 1804 ms
Custom cache: 10 model runs, 311 ms
Dono ke results same: True
{'size': 10, 'hits': 50, 'misses': 10, 'waits': 0, 'evictions': 0, 'expired': 0, 'hit_rate_%': 83.3}


## Task 3: Visualize prediction confidence using bar charts by manually computing probability distributions, implementing real-time updates with animations, and allowing users to interactively explore model confidence levels.

### Step 1: `confidence.py`

In [19]:
%%writefile confidence.py
"""
HeartCare - prediction confidence ko bar charts me dikhana.

1. Probability distribution HAATH se: Random Forest ki har tree ka probability nikal ke khud average karte hain
   (model.predict_proba ka use sirf check karne ke liye).
2. Charts HTML + CSS se bante hain. Har bar purani value se nayi value tak animate hota hai (CSS keyframes).
3. Interactive: threshold slider, what-if (ek feature badalke confidence kaise badalti hai).
"""
import numpy as np
import pandas as pd

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
CLASS_NAMES = {0: "Lower Risk", 1: "Higher Risk"}        # label badalna ho to sirf yahan
COLORS = {0: "#2e9e6b", 1: "#d64545"}
N_BINS = 10


# ---------------------------------------------------------------- 1. probability distribution (manual)
def analyse(model, x):
    """Ek patient ke liye poori confidence picture. Cache me yahi dict jati hai."""
    x32 = np.ascontiguousarray(x, dtype=np.float32).reshape(1, -1)               # trees float32 lete hain
    per_tree = np.array([t.predict_proba(x32, check_input=False)[0] for t in model.estimators_])
    probs = per_tree.mean(axis=0)                                                  # har class ki probability (manual average)
    votes = np.bincount(per_tree.argmax(axis=1), minlength=len(probs))             # kitni trees ne kaunsi class chuni
    hist = np.histogram(per_tree[:, 1], bins=N_BINS, range=(0, 1))[0]              # trees ke P(class 1) ka distribution
    return {"probs": probs, "votes": votes, "hist": hist, "n_trees": len(per_tree),
            "std": float(per_tree[:, 1].std()), "x": np.asarray(x, dtype=float)}


def decide(probs, threshold=0.5):
    """Threshold ke hisaab se class (default 0.5 = model.predict jaisa)."""
    return int(probs[1] > threshold)


def sweep(model, x, feature, values):
    """What-if: ek feature ki value badalke P(class 1) (ek hi predict_proba call me)."""
    rows = np.tile(np.asarray(x, dtype=float), (len(values), 1))
    rows[:, FEATURES.index(feature)] = values
    return model.predict_proba(pd.DataFrame(rows, columns=FEATURES))[:, 1]


def sweep_values(rule, n=10):
    """Rules (input.py) se feature ki values: category = allowed, number = n barabar points."""
    if rule["kind"] == "category":
        return [float(v) for v in rule["allowed"]]
    return [float(v) for v in np.round(np.linspace(rule["min"], rule["max"], n), 1)]


# ---------------------------------------------------------------- 2. HTML/CSS charts (animated)
CSS = """
<style>
.hc-chart { background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 16px; margin-bottom: 14px; }
.hc-chart h4 { margin: 0 0 12px 0; color: #142b53; font-size: 16px; }
.hc-row { display: flex; align-items: center; margin: 8px 0; }
.hc-label { width: 120px; font-size: 14px; color: #334155; }
.hc-track { position: relative; flex: 1; height: 22px; background: #eef2f7; border-radius: 6px; overflow: hidden; }
.hc-fill { height: 100%; width: var(--to); animation: hc-grow 0.9s ease-out; }
.hc-value { width: 64px; text-align: right; font-size: 14px; font-weight: 600; color: #142b53; }
.hc-thr { position: absolute; top: 0; bottom: 0; width: 2px; background: #142b53; }
.hc-cols { display: flex; align-items: flex-end; height: 130px; gap: 6px; border-bottom: 1px solid #cbd5e1; }
.hc-colwrap { flex: 1; height: 100%; display: flex; flex-direction: column; justify-content: flex-end; align-items: center; }
.hc-col { width: 100%; height: var(--to); border-radius: 4px 4px 0 0; animation: hc-rise 0.9s ease-out; }
.hc-colnum { font-size: 11px; color: #475569; height: 14px; }
.hc-xlabels { display: flex; gap: 6px; margin-top: 4px; }
.hc-xlabels span { flex: 1; text-align: center; font-size: 11px; color: #475569; }
.hc-note { font-size: 12px; color: #64748b; margin-top: 8px; }
@keyframes hc-grow { from { width: var(--from); } to { width: var(--to); } }
@keyframes hc-rise { from { height: var(--from); } to { height: var(--to); } }
@media (max-width: 600px) { .hc-label { width: 84px; font-size: 12px; } .hc-value { width: 52px; } }
@media (prefers-reduced-motion: reduce) { .hc-fill, .hc-col { animation: none; } }
</style>
"""


def _prev(prev, i, scale):
    """Purani value (animation ka start). Pehli baar 0 se grow hota hai."""
    return 0.0 if prev is None or i >= len(prev) else float(prev[i]) * scale


def bar_chart(title, labels, values, prev=None, colors=None, threshold=None, note=""):
    """Horizontal bars. values 0-1 (probability). threshold = vertical line."""
    rows = ""
    for i, (lab, v) in enumerate(zip(labels, values)):
        color = colors[i] if colors else "#244577"
        thr = f'<div class="hc-thr" style="left:{threshold * 100:.1f}%"></div>' if threshold is not None else ""
        rows += (f'<div class="hc-row"><div class="hc-label">{lab}</div>'
                 f'<div class="hc-track"><div class="hc-fill" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div>{thr}</div>'
                 f'<div class="hc-value">{v * 100:.1f}%</div></div>')
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4>{rows}{note_html}</div>'


def column_chart(title, x_labels, values, prev=None, color="#244577", counts=None, note=""):
    """Vertical bars. values 0-1 (height ka fraction); counts = bar ke upar likha number."""
    cols, labs = "", ""
    for i, (lab, v) in enumerate(zip(x_labels, values)):
        shown = counts[i] if counts is not None else f"{v * 100:.0f}%"
        cols += (f'<div class="hc-colwrap"><div class="hc-colnum">{shown}</div>'
                 f'<div class="hc-col" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div></div>')
        labs += f"<span>{lab}</span>"
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4><div class="hc-cols">{cols}</div><div class="hc-xlabels">{labs}</div>{note_html}</div>'

Writing confidence.py


### Step 2: Manual probability sahi hai ya nahi 

In [20]:
import numpy as np
import importlib, confidence
importlib.reload(confidence)
from confidence import *

X_all = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values
manual = np.array([analyse(model, x)['probs'] for x in X_all])
sk = model.predict_proba(pd.DataFrame(X_all, columns=FEATURES))
print('Patients:', len(X_all), '| max |manual - sklearn| =', np.abs(manual - sk).max())
print('Prediction same (threshold 50%):', all(decide(p) == model.classes_[k] for p, k in zip(manual, sk.argmax(1))))

Patients: 302 | max |manual - sklearn| = 0.0
Prediction same (threshold 50%): True


### Step 3: Ek patient ke charts (notebook me hi, animation ke saath)
Bar pehli baar 0 se grow hote hain. App me input badalne par purani value se nayi value tak animate hote hain.

In [21]:
from IPython.display import HTML

x = X_all[0]
info = analyse(model, x)
labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
print('Probabilities:', info['probs'], '| votes:', info['votes'], '| tree spread (std):', round(info['std'], 3))

HTML(CSS
     + bar_chart('Class probabilities (trees ka average)', labels, info['probs'], colors=[COLORS[0], COLORS[1]], threshold=0.5)
     + bar_chart('Tree votes', labels, info['votes'] / info['n_trees'], colors=[COLORS[0], COLORS[1]])
     + column_chart('Har tree ka Higher Risk probability (%)', [f'{i*10}-{i*10+10}' for i in range(10)],
                    info['hist'] / info['hist'].max(), counts=[int(h) for h in info['hist']]))

Probabilities: [0.22 0.78] | votes: [22 78] | tree spread (std): 0.414


### Step 4: Interactive exploration: threshold aur what-if

In [22]:
for thr in (0.3, 0.5, 0.7, 0.9):
    print(f'threshold {thr:.0%}: prediction = {CLASS_NAMES[decide(info["probs"], thr)]}')

rules = load_rules('heart.csv')
feature = 'cp'
values = sweep_values(rules[feature])
p_values = sweep(model, x, feature, values)
HTML(CSS + column_chart(f'What-if: {feature} badalne par {CLASS_NAMES[1]} probability', [f'{v:g}' for v in values], p_values, color='#d64545',
                        note='Baaki saari values wahi rakhi gayi hain; sirf ' + feature + ' badla.'))

threshold 30%: prediction = Higher Risk
threshold 50%: prediction = Higher Risk
threshold 70%: prediction = Higher Risk
threshold 90%: prediction = Lower Risk


## Final Dashboard: `app.py` 

In [23]:
%%writefile app.py

import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

from input import FEATURES, load_rules, validate_input, predict_batch      
from cache import PredictionCache
from confidence import (CSS, CLASS_NAMES, COLORS, analyse, decide, sweep,  
                        sweep_values, bar_chart, column_chart)

st.set_page_config(
    page_title="HeartCare",
    layout="centered"
)

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
MAX_BATCH_ROWS = 100000


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    return model, rules, cache, MODEL_FILE


try:
    model, rules, cache, model_version = load_everything()
except FileNotFoundError as e:
    st.error(f"File nahi mili: {e.filename}. app.py ke same folder me rakho.")
    st.stop()


def hint(f):
    r = rules[f]
    if r["kind"] == "category":
        return "/".join(str(int(a)) for a in r["allowed"])
    return f"{r['min']:g} to {r['max']:g}"


def choice(label, f):
    return st.selectbox(f"{label} ({hint(f)})", [None] + rules[f]["allowed"],
                        index=1, format_func=lambda v: "Not known" if v is None else str(int(v)))


def number(label, f, default):
    return st.text_input(f"{label} ({hint(f)})", value=default)


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


st.markdown("""
<style>
.stApp {
    background: #f3f6fb;
}

header, footer {
    visibility: hidden;
}

.main .block-container {
    max-width: 850px;
    padding: 25px 20px;
}

.header {
    background: #142b53;
    color: white;
    padding: 25px;
    border-radius: 14px;
    text-align: center;
    margin-bottom: 25px;
}

.section {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 18px;
    border: 1px solid #e2e8f0;
}

.section h3 {
    color: #142b53;
}

.stButton button {
    background: #142b53;
    color: white;
    width: 100%;
    height: 45px;
    border-radius: 8px;
}

.stButton button:hover {
    background: #244577;
    color: white;
}

@media(max-width: 600px) {
    .main .block-container {
        padding: 12px 10px;
    }

    .header {
        padding: 18px;
    }

    .header h1 {
        font-size: 24px;
    }

    .section {
        padding: 14px;
    }
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>HeartCare</h1>
    <p>Heart Disease Risk Prediction System</p>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="section">
    <h3>Patient Information</h3>
    <p>Enter patient medical details below.</p>
</div>
""", unsafe_allow_html=True)

st.caption("Koi field khali chhodo ya 'Not known' chuno to median/mode se bhar diya jayega (max 3 fields).")

age = number("Age", "age", "45")
sex = choice("Sex", "sex")
cp = choice("Chest Pain Type", "cp")
trestbps = number("Resting Blood Pressure", "trestbps", "120")
chol = number("Cholesterol", "chol", "200")
fbs = choice("Fasting Blood Sugar", "fbs")
restecg = choice("Resting ECG", "restecg")
thalach = number("Maximum Heart Rate", "thalach", "150")
exang = choice("Exercise Induced Angina", "exang")
oldpeak = number("Oldpeak", "oldpeak", "1.0")
slope = choice("Slope", "slope")
ca = choice("Major Vessels", "ca")
thal = choice("Thal", "thal")

live = st.checkbox("Live update (input badalte hi prediction update ho)")
clicked = st.button("Predict Heart Disease Risk")

# ---------------- Task 1: input validation
if clicked or live:

    record = dict(zip(FEATURES, [
        age, sex, cp, trestbps, chol,
        fbs, restecg, thalach, exang,
        oldpeak, slope, ca, thal
    ]))

    check = validate_input(record, rules)

    for message in check["errors"]:
        st.error(message)

    if check["ok"]:
        st.session_state["check"] = check       
    else:
        st.session_state.pop("check", None)

check = st.session_state.get("check")

if check:

    st.markdown(CSS, unsafe_allow_html=True)

    st.markdown("""
<div class="section">
    <h3>Prediction & Confidence</h3>
</div>
""", unsafe_allow_html=True)

    threshold = st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, 50) / 100
    explore = st.selectbox("What-if: ek feature badalke confidence dekho", ["(none)"] + FEATURES)

    # ---------------- Task 2: custom cache 
    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    # ---------------- Task 3: manually computed probabilities
    probs = info["probs"]
    prediction = decide(probs, threshold)

    if prediction == 1:
        st.error(f"{CLASS_NAMES[1]} Prediction")
    else:
        st.success(f"{CLASS_NAMES[0]} Prediction")

    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")

    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)

    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)

    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me",
                             bins, hist / hist.max(), prev=previous("hist", hist / hist.max()),
                             color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti. Sab trees ek jaisi ho to bars sirf kinaron par aate hain."),
                unsafe_allow_html=True)

    if explore != "(none)":
        values = sweep_values(rules[explore])
        p_values = sweep(model, check["x"], explore, values)
        st.markdown(column_chart(f"What-if: {explore} badalne par {CLASS_NAMES[1]} probability",
                                 [f"{v:g}" for v in values], p_values, prev=previous("sweep_" + explore, p_values),
                                 color="#d64545", note=f"Baaki saari values wahi rakhi gayi hain; sirf {explore} badla."),
                    unsafe_allow_html=True)

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")

    stats = cache.stats()
    st.caption(f"Is request me {elapsed:.2f} ms | Cache: {stats['hits']} hits, {stats['misses']} misses, "
               f"{stats['waits']} coalesced, hit rate {stats['hit_rate_%']}% (size {stats['size']})")

st.markdown("""
<div class="section">
    <h3>Batch Prediction (CSV)</h3>
    <p>Bahut patients ek saath: CSV me columns age, sex, cp, trestbps, chol, fbs,
    restecg, thalach, exang, oldpeak, slope, ca, thal hone chahiye.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])

data = None

if uploaded is not None:

    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:

    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]

    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}")

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000

        rejected = int((results["status"] == "rejected").sum())
        st.write(f"Rows: {len(data)} | Predicted: {len(data) - rejected} | Rejected: {rejected}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")

        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))

        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")

st.markdown("""
<div style="text-align:center;color:#64748b;padding:20px;">
    HeartCare | Educational Purpose Only
    <br>
    Not a substitute for professional medical diagnosis.
</div>
""", unsafe_allow_html=True)

Writing app.py


### Dashboard run 

In [ ]:
!streamlit run app.py